# 1. From a Task to a Flow: Silicon band structure

Every calculation in AbiPy is, at bottom, an `AbinitTask`: one `AbinitInput`
plus a working directory. `Task`s can depend on each other (e.g. a
non-self-consistent run needs the density from a self-consistent one), and
a `Flow` is what takes care of that bookkeeping for you -- ordering tasks
correctly, and letting you (re)launch the whole thing with one command.

This notebook builds the same calculation -- the band structure of silicon
-- three times, in increasingly automated ways. Each step corresponds to
one standalone script in `../Examples/`; rather than printing the whole
script, we'll look at the `workshop_lib.py` function(s) it's generated
from (with `wlib.print_source`), build the same object directly here,
then look at the result of actually running it (each has already been run
for you):

1. `run_si_gstate.py` -- a single Task (ground state).
2. `run_si_nscf.py` -- a second Task, depending on the first through a
   **hardcoded path**.
3. `run_si_ebands.py` -- both tasks registered in a `Flow`, with the
   dependency expressed **through the task object itself**.


In [20]:
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

from abipy import abilab
import abipy.flowtk as flowtk
abilab.enable_notebook()

%matplotlib inline

import workshop_lib as wlib

## Step 1: a single Task

The simplest possible unit of work: one `AbinitInput` wrapped in one
`AbinitTask`, built and run directly -- no `Flow`, no `Work`.


In [2]:
wlib.print_source(wlib.si_gs_input)
wlib.print_source(wlib.build_si_gs_task)

In [3]:
task = wlib.build_si_gs_task(workdir="../Examples/task_si_gstate")

<AbinitTask, node_id=1600, workdir=../Examples/task_si_gstate>

This exact `AbinitTask` is what `run_si_gstate.py` builds and runs from a
terminal (from inside `Examples/`, so the output directory lands there,
next to the script):

```
cd ../Examples
python run_si_gstate.py
```

This produces `task_si_gstate/`, including the density (`out_DEN.nc`) that
Step 2 depends on.


In [19]:
wlib.print_source(wlib.setup_task_manager)

In [14]:
wlib.setup_task_manager(task, mpi_procs=4, timelimit_hour=0.1)

# Remove a previous run, if it exists.
if Path(task.workdir).exists():
    print(f'Removing existing directory: {task.workdir}/')
    task.rmtree()

task.build()
task.make_links()

Removing existing directory: /Users/antonius/Documents/Workshops/2026-CEMDI/Tutorial/Examples/task_si_gstate/


In [16]:
task.start_and_wait()

0

In [17]:
task.check_status()

<Status: Completed, at 5390614480>

In [18]:
with abilab.abiopen("../Examples/task_si_gstate/outdata/out_GSR.nc") as gsr:
    print("energy:", gsr.energy, "eV")
    print(gsr.ebands)

energy: -229.77702155478488 eV eV
================================= Structure =================================
Full Formula (Si2)
Reduced Formula: Si
abc   :   3.849279   3.849279   3.849279
angles:  60.000000  60.000000  60.000000
pbc   :       True       True       True
Sites (2)
  #  SP        a      b      c  cartesian_forces
---  ----  -----  -----  -----  --------------------
  0  Si    0.875  0.875  0.875  [0. 0. 0.] eV ang^-1
  1  Si    0.125  0.125  0.125  [0. 0. 0.] eV ang^-1

Abinit Spacegroup: spgid: 227, num_spatial_symmetries: 48, has_timerev: True, symmorphic: True

Number of electrons: 8.0, Fermi level: 4.310 (eV)
nsppol: 1, nkpt: 29, mband: 16, nspinor: 1, nspden: 1
smearing scheme: none (occopt 1), tsmear_eV: 0.272, tsmear Kelvin: 3157.7
Direct gap:
    Energy: 2.581 (eV)
    Initial state: spin: 0, kpt: [+0.000, +0.000, +0.000], weight: 0.002, band: 3, eig: 4.310, occ: 2.000
    Final state:   spin: 0, kpt: [+0.000, +0.000, +0.000], weight: 0.002, band: 4, eig: 6.89

## Step 2: a second, dependent Task

Same idea, but this task's input is a band-structure (NSCF) calculation,
and it needs the density file produced in Step 1.


In [ ]:
wlib.print_source(wlib.si_bandstructure_input)
wlib.print_source(wlib.build_si_nscf_task)

In [ ]:
task = wlib.build_si_nscf_task(
    workdir="../Examples/task_si_nscf",
    density="../Examples/task_si_gstate/outdata/out_DEN.nc")
task

Notice the dependency: `deps = {density: 'DEN'}`, where `density` is just
the **string path** to Step 1's density file. This works, but it's
fragile -- rename or move `task_si_gstate/`, and this breaks with no
warning until Abinit complains about a missing file.

```
cd ../Examples
python run_si_nscf.py
```


In [ ]:
with abilab.abiopen("../Examples/task_si_nscf/outdata/out_GSR.nc") as gsr:
    ebands = gsr.ebands

fig = ebands.plot(color="b", show=False)
fig.gca().set_ylim(-12.5, 7.5);

## Step 3: wrapping both Tasks in a Flow

Same two tasks, same physics -- but now registered together in a `Work`
inside a `Flow`, with the dependency expressed as `deps={gs_task: 'DEN'}`:
a reference to the *task object* itself, not a path. `si_gs_input`,
`build_si_gs_task`, `si_bandstructure_input` and `build_si_nscf_task` are
exactly the same functions shown in Steps 1-2; the only new piece is how
they're wrapped:


In [ ]:
wlib.print_source(wlib.build_si_ebands_task_flow)
wlib.print_source(wlib.setup_manager)

```
cd ../Examples
python run_si_ebands.py
```

The `Flow` now knows the dependency graph, and can be inspected, resumed,
or re-run reliably, however many tasks it grows to. This loads the
*already-run* flow from disk (not `wlib.build_si_ebands_task_flow()`
directly, which would give us a fresh, not-yet-run copy):


In [ ]:
flow = flowtk.Flow.from_file("../Examples/flow_si_ebands")
flow.get_graphviz()

In [ ]:
task = flow[0][1]  # second task of the first Work: the NSCF run
gsr_path = task.outdir.has_abiext("GSR")

with abilab.abiopen(gsr_path) as gsr:
    ebands = gsr.ebands

fig = ebands.plot(color="b", show=False)
fig.gca().set_ylim(-12.5, 7.5);

Same plot as Step 2 -- same physics, computed with the same two Abinit
runs, just orchestrated by AbiPy instead of two manual scripts and a
hardcoded path.

> **Exercise.** Silicon's fundamental gap is indirect: the conduction band
> minimum is not at $\Gamma$. Zoom in on the plot above (or replot with a
> narrower `set_ylim`) to see it. We'll come back to this in the band
> structure section of `2-Existing_flows.ipynb`, comparing with GaAs --
> which has a *direct* gap at $\Gamma$.

Every flow from here on (Notebook 2 onward) is expressed as a `Flow` from
the start, for exactly this reason: it scales past two tasks without
becoming unmanageable.

Continue with [`2-Existing_flows.ipynb`](2-Existing_flows.ipynb).